In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from category_encoders import TargetEncoder

In [16]:
# =========================================================
# Path config
# =========================================================

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

SAVE_DIR = OOF_DIR / "combo_te_v1_clinical_missing_s10_seed42"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = SUB_DIR / "clinical_missing"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH:", TRAIN_PATH.resolve(), TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR.resolve())
print("SUB_SAVE_DIR:", SUB_SAVE_DIR.resolve())

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH: /mnt/c/dev/my_ml_project/data/train.csv True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_clinical_missing_s10_seed42
SUB_SAVE_DIR: /mnt/c/dev/my_ml_project/submissions/clinical_missing


In [17]:
def _to_num_safe(s):
    return pd.to_numeric(s, errors="coerce")


def _is_missing_category(s):
    s_str = s.astype(str).str.strip()
    return (
        s.isna() |
        s_str.isin(["", "nan", "NaN", "None", "알 수 없음"])
    )


def add_clinical_missing_inference_features(df):
    """
    임상 논리 기반 결측/프로세스 유추 feature.

    단일 valid에서 효과 있었던 feature:
    - DI_배아프로세스_구조적결측률
    - DI_배아프로세스_대부분결측
    - DI_배아프로세스_구조적결측수
    - 배란유도_임상추정
    - 임상근거_ICSI
    - 배란유도_자연주기추정
    - 임상근거_FER

    제거 후보:
    - 특정시술유형_결측여부
    - 특정시술유형_임상추정
    """
    df = df.copy()

    # =====================================================
    # 1. 특정 시술 유형 결측 여부
    # =====================================================
    if "특정 시술 유형" in df.columns:
        specific_missing = _is_missing_category(df["특정 시술 유형"])
    else:
        specific_missing = pd.Series(False, index=df.index)

    # 만들긴 하지만 마지막에 drop할 예정
    df["특정시술유형_결측여부"] = specific_missing.astype(int)

    # =====================================================
    # 2. FER / 냉동 배아 사용 근거
    # =====================================================
    thawed_embryo = (
        _to_num_safe(df["해동된 배아 수"]).fillna(0)
        if "해동된 배아 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    embryo_thaw_day_present = (
        df["배아 해동 경과일"].notna()
        if "배아 해동 경과일" in df.columns
        else pd.Series(False, index=df.index)
    )

    if "동결 배아 사용 여부" in df.columns:
        frozen_embryo_used = df["동결 배아 사용 여부"].fillna(0).astype(str).str.strip().isin(
            ["1", "1.0", "True", "true"]
        )
    else:
        frozen_embryo_used = pd.Series(False, index=df.index)

    df["임상근거_FER"] = (
        (thawed_embryo >= 1) |
        embryo_thaw_day_present |
        frozen_embryo_used
    ).astype(int)

    # =====================================================
    # 3. ICSI 근거
    # =====================================================
    micro_injected_egg = (
        _to_num_safe(df["미세주입된 난자 수"]).fillna(0)
        if "미세주입된 난자 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    micro_created_embryo = (
        _to_num_safe(df["미세주입에서 생성된 배아 수"]).fillna(0)
        if "미세주입에서 생성된 배아 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    micro_transfer_embryo = (
        _to_num_safe(df["미세주입 배아 이식 수"]).fillna(0)
        if "미세주입 배아 이식 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    df["임상근거_ICSI"] = (
        (micro_injected_egg >= 1) |
        (micro_created_embryo >= 1) |
        (micro_transfer_embryo >= 1)
    ).astype(int)

    # =====================================================
    # 4. 특정 시술 유형 결측 임상 추정 category
    # 단일 valid에서 FI 0이라 마지막에 drop할 예정
    # =====================================================
    df["특정시술유형_임상추정"] = "not_missing"

    df.loc[
        specific_missing,
        "특정시술유형_임상추정"
    ] = "missing_unknown"

    df.loc[
        specific_missing & (df["임상근거_FER"] == 1),
        "특정시술유형_임상추정"
    ] = "missing_inferred_FER"

    df.loc[
        specific_missing & (df["임상근거_ICSI"] == 1),
        "특정시술유형_임상추정"
    ] = "missing_inferred_ICSI"

    df.loc[
        specific_missing &
        (df["임상근거_FER"] == 1) &
        (df["임상근거_ICSI"] == 1),
        "특정시술유형_임상추정"
    ] = "missing_inferred_FER_ICSI"

    # =====================================================
    # 5. 배란 유도 자연주기 추정
    # =====================================================
    if "배란 유도 유형" in df.columns:
        ovulation_type_missing = _is_missing_category(df["배란 유도 유형"])
    else:
        ovulation_type_missing = pd.Series(False, index=df.index)

    if "배란 자극 여부" in df.columns:
        stimulation = df["배란 자극 여부"].astype(str).str.strip()
        no_stimulation = stimulation.isin(["0", "0.0", "False", "false", "아니오"])
    else:
        no_stimulation = pd.Series(False, index=df.index)

    df["배란유도_자연주기추정"] = (
        ovulation_type_missing &
        no_stimulation
    ).astype(int)

    if "배란 유도 유형" in df.columns:
        df["배란유도_임상추정"] = df["배란 유도 유형"].astype(str)
    else:
        df["배란유도_임상추정"] = "unknown"

    df.loc[
        df["배란유도_자연주기추정"] == 1,
        "배란유도_임상추정"
    ] = "자연주기_추정"

    # =====================================================
    # 6. DI 구조적 결측
    # =====================================================
    if "시술 유형" in df.columns:
        is_di = df["시술 유형"].astype(str).str.contains("DI", na=False)
    else:
        is_di = pd.Series(False, index=df.index)

    embryo_process_cols = [
        "난자 채취 경과일",
        "난자 해동 경과일",
        "난자 혼합 경과일",
        "배아 이식 경과일",
        "배아 해동 경과일",
        "총 생성 배아 수",
        "이식된 배아 수",
        "저장된 배아 수",
        "해동된 배아 수",
        "수집된 신선 난자 수",
        "혼합된 난자 수",
    ]

    existing_cols = [col for col in embryo_process_cols if col in df.columns]

    if existing_cols:
        missing_count = df[existing_cols].isna().sum(axis=1)
        missing_ratio = missing_count / len(existing_cols)

        df["DI_배아프로세스_구조적결측수"] = np.where(
            is_di,
            missing_count,
            0
        )

        df["DI_배아프로세스_구조적결측률"] = np.where(
            is_di,
            missing_ratio,
            0
        )

        df["DI_배아프로세스_대부분결측"] = (
            is_di &
            (missing_ratio >= 0.8)
        ).astype(int)
    else:
        df["DI_배아프로세스_구조적결측수"] = 0
        df["DI_배아프로세스_구조적결측률"] = 0
        df["DI_배아프로세스_대부분결측"] = 0

    return df

In [18]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [19]:
def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder


def make_cat_params(kind="main", seed=42):
    if kind == "main":
        return dict(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "shallow":
        return dict(
            iterations=2500,
            learning_rate=0.02,
            depth=5,
            l2_leaf_reg=8,
            random_strength=1.5,
            bagging_temperature=2,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "random":
        return dict(
            iterations=2500,
            learning_rate=0.022,
            depth=7,
            l2_leaf_reg=15,
            random_strength=8,
            bagging_temperature=8,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    raise ValueError(f"Unknown kind: {kind}")

def data_preprocessing_clinical_missing(df):
    df = df.copy()

    # fillna 전에 임상 결측/프로세스 추정 feature 생성
    df = add_clinical_missing_inference_features(df)

    # 기존 champion preprocessing 그대로 적용
    df = data_preprocessing(df)

    # 단일 valid FI 0이었던 feature 제거
    drop_clinical_cols = [
        "특정시술유형_결측여부",
        "특정시술유형_임상추정",
    ]

    df = df.drop(columns=drop_clinical_cols, errors="ignore")

    return df

def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder


def make_cat_params(kind="main", seed=42):
    if kind == "main":
        return dict(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "shallow":
        return dict(
            iterations=2500,
            learning_rate=0.02,
            depth=5,
            l2_leaf_reg=8,
            random_strength=1.5,
            bagging_temperature=2,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "random":
        return dict(
            iterations=2500,
            learning_rate=0.022,
            depth=7,
            l2_leaf_reg=15,
            random_strength=8,
            bagging_temperature=8,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    raise ValueError(f"Unknown kind: {kind}")

In [20]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]
X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

X = data_preprocessing_clinical_missing(X_raw)
X_test = data_preprocessing_clinical_missing(X_test_raw)

clinical_cols = [
    "DI_배아프로세스_구조적결측률",
    "DI_배아프로세스_대부분결측",
    "DI_배아프로세스_구조적결측수",
    "배란유도_임상추정",
    "임상근거_ICSI",
    "배란유도_자연주기추정",
    "임상근거_FER",
]

print("Clinical cols check")
print(X[clinical_cols].nunique())
display(X[clinical_cols].head())

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

# 첫 OOF에서는 clinical feature를 TE에 추가하지 않음
base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)
    X_test_stack[col] = X_test_stack[col].astype(str)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("cat_cols:", len(cat_cols))
print("combo cols remaining:", [c for c in X_stack.columns if c.endswith("_combo")])

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=FOLD_SEED
)

/tmp/ipykernel_535/835862898.py:209: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns
/tmp/ipykernel_535/835862898.py:209: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes fo

Clinical cols check
DI_배아프로세스_구조적결측률    2
DI_배아프로세스_대부분결측     2
DI_배아프로세스_구조적결측수    2
배란유도_임상추정           5
임상근거_ICSI           2
배란유도_자연주기추정         2
임상근거_FER            2
dtype: int64


,DI_배아프로세스_구조적결측률,DI_배아프로세스_대부분결측,DI_배아프로세스_구조적결측수,배란유도_임상추정,임상근거_ICSI,배란유도_자연주기추정,임상근거_FER
0,0.0,0,0,기록되지 않은 시행,1,0,0
1,0.0,0,0,자연주기_추정,1,1,0
2,0.0,0,0,기록되지 않은 시행,0,0,0
3,0.0,0,0,기록되지 않은 시행,1,0,0
4,0.0,0,0,기록되지 않은 시행,1,0,0


base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 117)
X_test_stack: (90067, 117)
cat_cols: 55
combo cols remaining: []


In [21]:
def train_cat_seed_ensemble(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    seeds,
    kind="main",
    name="main_cat"
):
    seed_oof_list = []
    seed_test_list = []
    score_rows = []

    for seed in seeds:
        print(f"\n================ {name} seed {seed} ================")

        oof = np.zeros(len(X_stack))
        test_pred = np.zeros(len(X_test_stack))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
            print(f"\n{name} seed {seed} / Fold {fold}")

            X_tr = X_stack.iloc[tr_idx].copy()
            X_val = X_stack.iloc[val_idx].copy()
            y_tr = y_stack.iloc[tr_idx]
            y_val = y_stack.iloc[val_idx]

            model = CatBoostClassifier(**make_cat_params(kind=kind, seed=seed))

            early_stop = 100 if kind == "main" else 150

            model.fit(
                X_tr,
                y_tr,
                cat_features=cat_cols,
                eval_set=(X_val, y_val),
                early_stopping_rounds=early_stop,
                verbose=100
            )

            val_pred = model.predict_proba(X_val)[:, 1]
            oof[val_idx] = val_pred

            fold_auc = roc_auc_score(y_val, val_pred)
            print(f"{name} seed {seed} Fold {fold} AUC:", fold_auc)

            score_rows.append({
                "model": name,
                "seed": seed,
                "fold": fold,
                "auc": fold_auc
            })

            test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        seed_auc = roc_auc_score(y_stack, oof)
        print(f"\n{name} seed {seed} OOF AUC:", seed_auc)

        score_rows.append({
            "model": name,
            "seed": seed,
            "fold": "OOF",
            "auc": seed_auc
        })

        seed_oof_list.append(oof)
        seed_test_list.append(test_pred)

        np.save(SAVE_DIR / f"{name}_seed{seed}_oof.npy", oof)
        np.save(SAVE_DIR / f"{name}_seed{seed}_test.npy", test_pred)

    final_oof = np.mean(seed_oof_list, axis=0)
    final_test = np.mean(seed_test_list, axis=0)

    final_auc = roc_auc_score(y_stack, final_oof)
    print(f"\n{name} seed ensemble OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": "ensemble",
        "fold": "OOF",
        "auc": final_auc
    })

    return final_oof, final_test, pd.DataFrame(score_rows)


def train_cat_single(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    kind="shallow",
    name="shallow_cat"
):
    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ {name} Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        model = CatBoostClassifier(**make_cat_params(kind=kind, seed=42))

        early_stop = 150 if kind == "shallow" else 200

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=early_stop,
            verbose=100
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"{name} Fold {fold} AUC:", fold_auc)

        score_rows.append({
            "model": name,
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"{name}_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"{name}_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print(f"\n{name} OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)

In [22]:
main_cat_oof, main_cat_test_pred, main_score_df = train_cat_seed_ensemble(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    seeds=[42, 77, 2024],
    kind="main",
    name="main_cat"
)

np.save(SAVE_DIR / "main_cat_oof.npy", main_cat_oof)
np.save(SAVE_DIR / "main_cat_test_pred.npy", main_cat_test_pred)


shallow_cat_oof, shallow_cat_test_pred, shallow_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="shallow",
    name="shallow_cat"
)

np.save(SAVE_DIR / "shallow_cat_oof.npy", shallow_cat_oof)
np.save(SAVE_DIR / "shallow_cat_test_pred.npy", shallow_cat_test_pred)


random_cat_oof, random_cat_test_pred, random_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="random",
    name="random_cat"
)

np.save(SAVE_DIR / "random_cat_oof.npy", random_cat_oof)
np.save(SAVE_DIR / "random_cat_test_pred.npy", random_cat_test_pred)


================ main_cat seed 42 ================

main_cat seed 42 / Fold 1
0:	test: 0.7281964	best: 0.7281964 (0)	total: 431ms	remaining: 14m 22s
100:	test: 0.7358493	best: 0.7358493 (100)	total: 13.8s	remaining: 4m 19s
200:	test: 0.7374874	best: 0.7374874 (200)	total: 43.3s	remaining: 6m 27s
300:	test: 0.7378565	best: 0.7378710 (295)	total: 56.4s	remaining: 5m 18s
400:	test: 0.7380460	best: 0.7380460 (400)	total: 1m 8s	remaining: 4m 31s
500:	test: 0.7381402	best: 0.7381435 (498)	total: 1m 19s	remaining: 3m 57s
600:	test: 0.7382047	best: 0.7382116 (595)	total: 1m 31s	remaining: 3m 32s
700:	test: 0.7382425	best: 0.7382430 (699)	total: 1m 43s	remaining: 3m 11s
800:	test: 0.7381942	best: 0.7382445 (701)	total: 1m 55s	remaining: 2m 52s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7382444796
bestIteration = 701

Shrink model to first 702 iterations.
main_cat seed 42 Fold 1 AUC: 0.7382444796454251

main_cat seed 42 / Fold 2
0:	test: 0.7283566	best: 0.7283566 (0)	t

In [23]:
main_cat_oof, main_cat_test_pred, main_score_df = train_cat_seed_ensemble(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    seeds=[42, 77, 2024],
    kind="main",
    name="main_cat"
)

np.save(SAVE_DIR / "main_cat_oof.npy", main_cat_oof)
np.save(SAVE_DIR / "main_cat_test_pred.npy", main_cat_test_pred)


shallow_cat_oof, shallow_cat_test_pred, shallow_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="shallow",
    name="shallow_cat"
)

np.save(SAVE_DIR / "shallow_cat_oof.npy", shallow_cat_oof)
np.save(SAVE_DIR / "shallow_cat_test_pred.npy", shallow_cat_test_pred)


random_cat_oof, random_cat_test_pred, random_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="random",
    name="random_cat"
)

np.save(SAVE_DIR / "random_cat_oof.npy", random_cat_oof)
np.save(SAVE_DIR / "random_cat_test_pred.npy", random_cat_test_pred)


================ main_cat seed 42 ================

main_cat seed 42 / Fold 1
0:	test: 0.7281964	best: 0.7281964 (0)	total: 135ms	remaining: 4m 29s
100:	test: 0.7358493	best: 0.7358493 (100)	total: 12.1s	remaining: 3m 46s
200:	test: 0.7374874	best: 0.7374874 (200)	total: 23.6s	remaining: 3m 31s
300:	test: 0.7378565	best: 0.7378710 (295)	total: 35.1s	remaining: 3m 17s
400:	test: 0.7380460	best: 0.7380460 (400)	total: 46.3s	remaining: 3m 4s
500:	test: 0.7381402	best: 0.7381435 (498)	total: 57.3s	remaining: 2m 51s
600:	test: 0.7382047	best: 0.7382116 (595)	total: 1m 9s	remaining: 2m 40s
700:	test: 0.7382425	best: 0.7382430 (699)	total: 1m 20s	remaining: 2m 29s
800:	test: 0.7381942	best: 0.7382445 (701)	total: 1m 32s	remaining: 2m 17s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7382444796
bestIteration = 701

Shrink model to first 702 iterations.
main_cat seed 42 Fold 1 AUC: 0.7382444796454251

main_cat seed 42 / Fold 2
0:	test: 0.7283566	best: 0.7283566 (0)	total

In [24]:
def train_xgb_oof_test(X_stack, X_test_stack, y_stack, skf):
    numeric_features = X_stack.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ XGB Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        X_tr_trans = preprocessor.fit_transform(X_tr)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test_stack)

        model = XGBClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            scale_pos_weight=POS_WEIGHT,
            tree_method="hist",
            n_jobs=-1
        )

        model.fit(
            X_tr_trans,
            y_tr.to_numpy().ravel(),
            eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
            verbose=False
        )

        val_pred = model.predict_proba(X_val_trans)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print("XGB Fold AUC:", fold_auc)

        score_rows.append({
            "model": "xgb",
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_trans)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"xgb_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"xgb_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print("\nXGB OOF AUC:", final_auc)

    score_rows.append({
        "model": "xgb",
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)


xgb_oof, xgb_test_pred, xgb_score_df = train_xgb_oof_test(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    skf=skf
)

np.save(SAVE_DIR / "xgb_oof.npy", xgb_oof)
np.save(SAVE_DIR / "xgb_test_pred.npy", xgb_test_pred)


================ XGB Fold 1 ================
XGB Fold AUC: 0.7361220362208759

================ XGB Fold 2 ================
XGB Fold AUC: 0.7394852207511846

================ XGB Fold 3 ================
XGB Fold AUC: 0.7386345515370693

================ XGB Fold 4 ================
XGB Fold AUC: 0.7371847261275006

================ XGB Fold 5 ================
XGB Fold AUC: 0.7389029623494228

XGB OOF AUC: 0.7379529030241652


In [25]:
main_cat_rank_oof = rank01(main_cat_oof)
shallow_cat_rank_oof = rank01(shallow_cat_oof)
random_cat_rank_oof = rank01(random_cat_oof)
xgb_rank_oof = rank01(xgb_oof)

final_oof_clinical_seed42 = (
    0.44 * main_cat_rank_oof +
    0.08 * shallow_cat_rank_oof +
    0.35 * random_cat_rank_oof +
    0.13 * xgb_rank_oof
)

final_oof_auc = roc_auc_score(y_stack, final_oof_clinical_seed42)

print("\n====================")
print("Clinical Missing Seed42 Main Cat OOF:", roc_auc_score(y_stack, main_cat_oof))
print("Clinical Missing Seed42 Shallow Cat OOF:", roc_auc_score(y_stack, shallow_cat_oof))
print("Clinical Missing Seed42 Random Cat OOF:", roc_auc_score(y_stack, random_cat_oof))
print("Clinical Missing Seed42 XGB OOF:", roc_auc_score(y_stack, xgb_oof))
print("Clinical Missing Seed42 Final Rank Blend OOF:", final_oof_auc)

np.save(SAVE_DIR / "final_oof_clinical_missing_seed42.npy", final_oof_clinical_seed42)
np.save(SAVE_DIR / "y_stack.npy", y_stack.to_numpy())

score_df = pd.concat(
    [main_score_df, shallow_score_df, random_score_df, xgb_score_df],
    ignore_index=True
)

score_df.to_csv(SAVE_DIR / "fold_oof_scores.csv", index=False)

summary_df = pd.DataFrame([
    {
        "fold_seed": FOLD_SEED,
        "feature_set": "combo_te_v1_s10_clinical_missing",
        "main_cat_oof": roc_auc_score(y_stack, main_cat_oof),
        "shallow_cat_oof": roc_auc_score(y_stack, shallow_cat_oof),
        "random_cat_oof": roc_auc_score(y_stack, random_cat_oof),
        "xgb_oof": roc_auc_score(y_stack, xgb_oof),
        "final_rank_blend_oof": final_oof_auc,
        "weights": "main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13",
        "note": "Clinical missing inference added. Clinical features not added to TE cols."
    }
])

summary_df.to_csv(SAVE_DIR / "summary.csv", index=False)
display(summary_df)


Clinical Missing Seed42 Main Cat OOF: 0.7402877825685538
Clinical Missing Seed42 Shallow Cat OOF: 0.7401996506150448
Clinical Missing Seed42 Random Cat OOF: 0.740160266797603
Clinical Missing Seed42 XGB OOF: 0.7379529030241652
Clinical Missing Seed42 Final Rank Blend OOF: 0.7403811774120048


,fold_seed,feature_set,main_cat_oof,shallow_cat_oof,random_cat_oof,xgb_oof,final_rank_blend_oof,weights,note
0,42,combo_te_v1_s10_clinical_missing,0.740288,0.7402,0.74016,0.737953,0.740381,main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13,Clinical missing inference added. Clinical fea...


In [26]:
main_cat_rank_test = rank01(main_cat_test_pred)
shallow_cat_rank_test = rank01(shallow_cat_test_pred)
random_cat_rank_test = rank01(random_cat_test_pred)
xgb_rank_test = rank01(xgb_test_pred)

final_pred_clinical_missing_seed42 = (
    0.44 * main_cat_rank_test +
    0.08 * shallow_cat_rank_test +
    0.35 * random_cat_rank_test +
    0.13 * xgb_rank_test
)

assert len(final_pred_clinical_missing_seed42) == len(submission)
assert np.isfinite(final_pred_clinical_missing_seed42).all()
assert final_pred_clinical_missing_seed42.min() >= 0
assert final_pred_clinical_missing_seed42.max() <= 1

pred_col = submission.columns[-1]

submission_clinical = submission.copy()
submission_clinical[pred_col] = final_pred_clinical_missing_seed42

submission_clinical.to_csv(
    SUB_SAVE_DIR / "submission_combo_te_v1_clinical_missing_seed42.csv",
    index=False
)

np.save(
    SUB_SAVE_DIR / "final_pred_combo_te_v1_clinical_missing_seed42.npy",
    final_pred_clinical_missing_seed42
)

print("\nClinical missing seed42 저장 완료")
print(submission_clinical[pred_col].describe())


Clinical missing seed42 저장 완료
count    90067.000000
mean         0.500006
std          0.288318
min          0.000034
25%          0.250541
50%          0.500400
75%          0.749505
max          0.999991
Name: probability, dtype: float64
